**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Diffusion Models

The generative method behind modern image synthesis, told as a *signal processing* story: corrupt data with Gaussian noise step by step, train a network to **denoise**, then run the corruption in reverse. We train a complete diffusion model on 2-D data in minutes and watch noise crystallize into structure.

## 1. Pre-requisites

- [Intro to PyTorch](../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb).
- [Random Variables](../Intro_Math/Analysis/Random_Variables.ipynb) (Gaussians compose).
- [Representation Learning](./Representation_Learning.ipynb) S2 for the generative-model context.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

# target distribution: two moons — structured, multimodal, low-dimensional
def moons(n):
    t = rng.uniform(0, np.pi, n)
    top = np.stack([np.cos(t), np.sin(t)], 1)
    bot = np.stack([1 - np.cos(t), 0.4 - np.sin(t)], 1)
    X = np.concatenate([top[:n//2], bot[n//2:]]) + 0.06*rng.standard_normal((n, 2))
    return ((X - X.mean(0)) / X.std(0)).astype(np.float32)

X = torch.from_numpy(moons(6000))
plt.figure(figsize=(3.6, 3.2)); plt.scatter(*X.T, s=2, alpha=0.3)
plt.title("the distribution we want to SAMPLE from"); plt.axis("equal")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2694033/3980811452.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 1 of 2 — *The Forward Process & the Denoising Objective* (~35 min)
**Goal:** destroy data with scheduled noise; train a network to predict the noise.
**Feeds into:** Session 2 (sampling = reverse diffusion).

---

## 2. Destruction Is Easy — Learn to Undo It

💡 **Intuition.** Generating from scratch is hard; *removing a little noise* is easy — it's [Wiener denoising's](../Intro_DSP/Statistical_Signal_Processing.ipynb) cousin, a regression problem. Diffusion's insight: chain the easy problem. Define a forward process that gradually noises data into pure Gaussian ($x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$ — Gaussians compose, so any step is one formula), and train one network $\varepsilon_\theta(x_t, t)$ to **predict the noise** that was added. Denoising at *every* noise level = knowing the path from chaos back to data.

In [2]:
T = 200
betas = torch.linspace(1e-4, 0.04, T)
alphas = 1 - betas
abar = torch.cumprod(alphas, 0)                      # ᾱ_t

# visualize the forward death of the data
fig, axes = plt.subplots(1, 4, figsize=(10, 2.4))
for ax, t_show in zip(axes, [0, 60, 120, 199]):
    eps = torch.randn_like(X)
    xt = abar[t_show].sqrt()*X + (1-abar[t_show]).sqrt()*eps
    ax.scatter(*xt.T, s=1, alpha=0.2); ax.set_title(f"t={t_show}"); ax.axis("equal"); ax.set_xlim(-3,3); ax.set_ylim(-3,3)
plt.suptitle("forward process: structure dissolves into N(0, I)")
plt.tight_layout(); plt.show()

Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.


Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


/tmp/ipykernel_2694033/3661612610.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [3]:
# the denoiser: predicts ε from (x_t, t) — t is embedded sinusoidally, like a transformer position
class Denoiser(nn.Module):
    def __init__(self, d=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2 + 16, d), nn.SiLU(), nn.Linear(d, d), nn.SiLU(),
                                 nn.Linear(d, d), nn.SiLU(), nn.Linear(d, 2))
    def t_embed(self, t):
        k = torch.arange(8)
        ang = t[:, None] / T * (100 ** (k / 8))[None]
        return torch.cat([ang.sin(), ang.cos()], 1)
    def forward(self, x, t):
        return self.net(torch.cat([x, self.t_embed(t.float())], 1))

model = Denoiser()
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
for step in range(4000):
    ix = torch.randint(0, len(X), (256,))
    x0 = X[ix]
    t = torch.randint(0, T, (256,))
    eps = torch.randn_like(x0)
    xt = abar[t, None].sqrt()*x0 + (1-abar[t, None]).sqrt()*eps
    loss = ((model(xt, t) - eps)**2).mean()            # predict the noise. that's the whole loss.
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 1000 == 0: print(f"step {step:4d}  denoising loss {loss.item():.4f}")

step    0  denoising loss 0.9189


step 1000  denoising loss 0.3236


step 2000  denoising loss 0.4127


step 3000  denoising loss 0.3509


---
### 🕐 Session 2 of 2 — *Sampling: Running Time Backwards* (~40 min)
**Goal:** start from pure noise and iteratively denoise into fresh samples.
**Builds on:** Session 1.

---

## 3. The Reverse Process

💡 **Intuition.** To sample: start at $x_T \sim \mathcal{N}(0, I)$ and repeatedly apply the learned denoiser, stepping $t = T{-}1, \dots, 0$, re-injecting a *little* fresh noise each step (the stochasticity keeps samples diverse — drop it and you get DDIM's deterministic cousin). Each step is a small, easy denoise; a few hundred of them compound into creation. Image generators are this exact loop with a U-Net denoiser and billions of pixels.

In [4]:
@torch.no_grad()
def sample(n_samp=3000, snapshots=(199, 120, 60, 0)):
    x = torch.randn(n_samp, 2)
    shots = {}
    for t_i in reversed(range(T)):
        t = torch.full((n_samp,), t_i)
        eps_hat = model(x, t)
        x0_hat = (x - (1-abar[t_i]).sqrt()*eps_hat) / abar[t_i].sqrt()
        if t_i > 0:
            ab_prev = abar[t_i-1]
            x = ab_prev.sqrt()*x0_hat + (1-ab_prev-betas[t_i]).clamp(min=0).sqrt()*eps_hat                 + betas[t_i].sqrt()*torch.randn_like(x)
        else:
            x = x0_hat
        if t_i in snapshots: shots[t_i] = x.clone()
    return shots

shots = sample()
fig, axes = plt.subplots(1, 4, figsize=(10, 2.4))
for ax, (t_i, xs) in zip(axes, sorted(shots.items(), reverse=True)):
    ax.scatter(*xs.T, s=1, alpha=0.2); ax.set_title(f"t={t_i}")
    ax.axis("equal"); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
plt.suptitle("reverse process: noise crystallizes into the two moons")
plt.tight_layout(); plt.show()

Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.


/tmp/ipykernel_2694033/2917646786.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [5]:
# quantitative check: do generated samples match the data's statistics?
gen = shots[0]
def stats(A):
    A = np.asarray(A)
    return A.mean(0).round(2), np.cov(A.T).round(2)
m_d, C_d = stats(X); m_g, C_g = stats(gen)
print("data  mean", m_d, " cov\n", C_d)
print("model mean", m_g, " cov\n", C_g)
# and the harder test: fraction of generated points close to the true manifold
from scipy.spatial import cKDTree
dist, _ = cKDTree(X.numpy()).query(gen.numpy())
print(f"\n90th-percentile distance of generated points to the data manifold: {np.quantile(dist, 0.9):.3f}")

data  mean [ 0. -0.]  cov
 [[ 1.   -0.47]
 [-0.47  1.  ]]
model mean [0.04 0.02]  cov
 [[ 1.06 -0.52]
 [-0.52  1.02]]

90th-percentile distance of generated points to the data manifold: 0.040


**The DSP lens, explicitly:** the forward process is progressive low-pass-plus-noise (coarse structure survives longest); the reverse process therefore builds coarse structure first and details last — generation as *spectral refinement*. That's also why diffusion models are natural denoisers, inpainters, and super-resolvers: those are just partial trips along the same chain.

## 4. Conclusion

One regression loss (predict the noise), one schedule, and a walk backwards through it: that's the entire method behind modern generative imagery — and you just trained one.

---
## Where next

- [Representation Learning](./Representation_Learning.ipynb) — VAEs: the previous generation of generation.
- [Statistical Signal Processing](../Intro_DSP/Statistical_Signal_Processing.ipynb) — the denoising theory underneath.
- [Scaling Neural Networks](./Scale_NN/Scale_NN.ipynb) — what it takes to run this at image scale.